In [ ]:
from scipy.optimize import brentq 
def f(x):
    F=x**2+3*x-18    
    return(F)
root=brentq(f,-1,5)
print(root)

In [ ]:
from scipy.optimize import brentq
import numpy as np
A1=6.90565
B1=1211.033
C1=220.79
A2=6.95464
B2=1344.8
C2=219.48
def dew_eq(T):
    P_sat=[10**(A1-B1/(C1+T)),10**(A2-B2/(C2+T))]
    yi=[0.4,0.6]
    P=760 
    return sum(y*P/P_vap for y,P_vap in zip(yi,P_sat))-1
print(dew_eq(80))
T_dew=brentq(dew_eq,50,150)
print(T_dew)

    

In [ ]:
from scipy.integrate import quad
def Heat_Capacity(T):
    return(25+0.01*T+10e-5*T**2)
dH,error=quad(Heat_Capacity,200,300)
print(dH)
print(error)
def Pressure(V):
    n=3
    R=8.314
    T=300
    return(n*R*T/V)
P,error=quad(Pressure,0.12,0.8)
print(P)
print(error)
    

In [ ]:
from scipy.integrate import odeint 
import numpy as np
import matplotlib.pyplot as plt 
t=np.linspace(0,10,100)
def mango(Ca,t):
    k=0.5
    dCadt=-k*Ca
    return dCadt
Ca0=1
solution=odeint(mango,Ca0,t)
plt.plot(t,solution,label=f"1st order kinetics with {solution[99]} as final concentration")
plt.xlabel('Time(min)')
plt.ylabel('concentration of A (mol/L)')
plt.title('Batch Reactor')
plt.legend()
plt.show()

In [ ]:
from scipy.integrate import odeint 
import numpy as np
import matplotlib.pyplot as plt 
t=np.linspace(0,10,100)
def reactor(y,t):
    Ca,Cb=y
    k=0.5
    dCadt=-k*Ca
    dCbdt=k*Ca
    return [dCadt,dCbdt]
y0=[1,0]
solution=odeint(reactor,y0,t)
Ca = solution[:,0]
Cb = solution[:,1]
plt.plot(t, Ca, label='A')
plt.plot(t, Cb, label='B')
plt.xlabel('Time (min)')
plt.ylabel('Concentration (mol/L)')
plt.title('Batch Reactor A → B')
plt.legend()
plt.show()


    

In [ ]:
from scipy.integrate import odeint 
import numpy as np
import matplotlib.pyplot as plt 
t=np.linspace(0,10,100)
def reactor(y,t,T):
    Ca,Cb=y
    Af = 1e7
    Ar=1e6
    Ef = 50000
    Er=60000
    R = 8.314
    kf=Af*np.exp(-Ef/(R*T))
    kr=Ar*np.exp(-Er/(R*T))
    Rf=kf*Ca
    Rr=kr*Cb
    dCadt=-Rf+Rr
    dCbdt=Rf-Rr
    return np.array([dCadt,dCbdt])
Temps=np.array([350,400,450])
Styles=np.array(['-','--','-.'])
y0=[1,0]
fig,ax=plt.subplots(2,1)
for Temps,Styles in zip(Temps,Styles):
    Solution=odeint(reactor,y0,t,args=(Temps,))
    CA=Solution[:,0]
    CB=Solution[:,1]
    ax[0].plot(t,CA,Styles,label=f'T={Temps}K')
    ax[1].plot(t,CB,Styles,label=f'T={Temps}K')
    ax[0].legend()
    ax[1].legend()
ax[0].set_xlabel('Time')
ax[1].set_xlabel('Time')
ax[0].set_ylabel('CA')
ax[1].set_ylabel('CB')
ax[0].set_title('Change in concentrations in a reversible reaction')
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
time_exp=[0,1,2,3,4,5,6]
conv_exp=[0.00,0.22,0.40,0.57,0.69,0.79,0.86]
data=pd.DataFrame({'Time':time_exp,'Conversion':conv_exp})
data.head()
data.describe()
plt.scatter(data['Time'],data['Conversion'],label='Experimental')
t_model=np.linspace(0,10,100)
k=0.3
conv_model=1-np.exp(-k*np.array(t_model))
plt.plot(t_model,conv_model,'--',label='model')
plt.xlabel('time')
plt.ylabel('Conversion')
plt.title('Model vs Experimental data')
plt.legend()
plt.show()


In [ ]:
#"Steady State CSTR with Recycle" Mini project
#Assumptions:Constant density,Constant volumetric flow rate,Perfect mixing,Isothermal operation,First-order irreversible reaction,No pressure drop
from scipy.optimize import fsolve
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np 
Qf=100 #Volumetric flow rate of fresh feed in L/min
Ca0=2 #Fresh feed concentration in mol/L
V=500 #Total reactor volume in L
k=0.25 #Rate constant (1/min for 1st order)
R=2 #Recycle Ratio =Qr/Qf
Qr=R*Qf
Qt=Qf+Qr #Total flow rate entering the reactor 
#Through the balalnce eqn. Qf*Ca0+Qr*Ca=Qt*Cin and we get, Cin=Qf*Ca0+Qr*Cout/Qt
def balance(Cout):
    Cout_scalar = float(Cout[0]) 
    # Mixing point calculation
    Cin=(Qf*Ca0+Qr*Cout_scalar )/Qt
    # For a 1st order reaction: Rate = k * Cout
    return Qt*Cin-Qt*Cout_scalar -V*k*Cout
COUT=fsolve(balance,1)
Cout_val=COUT[0]
#Calculate conversion
X=(Ca0-Cout_val)/Ca0
#Percent conversion
X_per=X*100
print(f'The concentration in the reactor and the outlet is {Cout_val:.4f} mol/L')
print(f'Overall percent conversion (X) is {X_per:.2f}%')
Recycle_ratios=np.array([0,1,2,3,4,5])
X_per_values=[]
for R in Recycle_ratios:
    Qr=R*Qf
    Qt=Qf+Qr #Total flow rate entering the reactor 
    #Through the balalnce eqn. Qf*Ca0+Qr*Ca=Qt*Cin and we get, Cin=Qf*Ca0+Qr*Cout/Qt
    def balance(Cout):
        Cout_scalar = float(Cout[0])
        # Mixing point calculation
        Cin=(Qf*Ca0+Qr*Cout_scalar )/Qt
        # For a 1st order reaction: Rate = k * Cout
        return Qt*Cin-Qt*Cout_scalar -V*k*Cout
    COUT=fsolve(balance,1)
    #Calculate conversion
    X=(Ca0-COUT)/Ca0
    #Percent conversion
    X_per=X*100 
    X_per_values.append(X_per)
fig,ax=plt.subplots()
for R_val,X_val in zip(Recycle_ratios,X_per_values):
    ax.scatter(R_val,X_val)
ax.set_xlabel('Recycle Ratios')
ax.set_ylabel('Percent Conversion')
ax.set_title('CSTR conversion — first-order reaction')
plt.tight_layout()
plt.show()
#Discussion: The conversion remained constant for all recycle ratios investigated.
#Examination of the steady-state material balance shows that the recycle terms cancel algebraically because the recycle stream has the same composition as the reactor outlet.    
#Under the assumptions of perfect mixing, constant density, and direct recycle without separation, recycle does not influence reactor conversion.    



    
    




    
